# 07 · Soil properties and conservation practices against the drought penalty (pre-registered)

**Question.** Do surface organic matter (primary), other soil properties (exploratory) and 2017 conservation-practice shares (exploratory) go with smaller drought penalties in Iowa, using the notebook 05 framework unchanged?

**Status.** This cell is the full plan, written and committed **before any result exists**. Every rule below is fixed in advance and is not changed after seeing results. Whatever the outcome, it is reported. Iowa only; corn and soybeans separately; `fips` kept as 5-character text.

## Framework (imported, not copied; `soil_tool.py` is not edited)
- Drought penalty and placebo: `soil_tool.compute_penalties(load_iowa_with_lon(), split=True)`, exactly as in notebook 05 Step 2 (normals never cross 2015/2016; drought rows are D2+, `max_drought_severity >= 3`; non-D2+ rows carry the placebo value). Parameters asserted from the module at run time: `WINDOW = 3`, `MIN_NORMAL_YEARS = 2`, `SEV_D2 = 3`, `MIN_COUNTIES_PER_YEAR = 15` (years with at least 15 county-years of the relevant kind with a value), `N_BOOT = 5000`, `BOOT_SEED = 20261017`, 95% range from the 2.5th and 97.5th percentiles of year-bootstrap means. Nothing is re-derived.
- Per-year regression and bootstrap: `soil_tool.penalty_storage_results` (per year, OLS slope of the penalty on the variable plus controls across that year's counties, averaged over years, year-bootstrap range; separately for drought rows and placebo rows). It is hard-wired to regress on the storage column `aws_100cm_mm` in units of 10 mm (`AWS_UNIT`) with the specifications in `PEN_SPECS`. **Thin wrapper, declared now:** the notebook function `run_variable(ia_split, variable, specs, unit)` copies the variable of interest into the `aws_100cm_mm` column of a *copy* of the data (the real storage values are kept under another name and are never used as a control), sets the module attributes `AWS_UNIT = unit` and `PEN_SPECS = specs` for the duration of the call, calls `penalty_storage_results`, and restores the attributes. Year selection, bootstrap, placebo and output format are therefore the module's own.
- Categorical variable (dominant drainage class): `penalty_storage_results` regresses on one variable, so the wrapper `run_categorical` fits, per year, one OLS of the penalty on the class dummies plus the location controls, with the **same** year rule (`MIN_COUNTIES_PER_YEAR`), the same drought/placebo split and the module's own `_year_bootstrap` for each dummy coefficient. Reference class: **Well drained**.
- County variables: surface organic matter, clay, sand, slope and dominant drainage class from `soil_planner/extra_data/iowa_soil_extra.csv`; 2017 practice shares from `soil_planner/extra_data/iowa_conservation_practices.csv`; `aws_100cm_mm` from the master; county centroid **latitude and longitude** computed from `load_counties()` as the mean of the polygon vertices (the same construction as `soil_tool.county_centroid_lon`, extended to latitude).

## Part 1: PRIMARY TEST, declared in advance: surface organic matter
1. Descriptives first: Pearson correlations of organic matter with longitude, latitude, slope and `aws_100cm_mm` across the 99 counties, and mean organic matter by dominant drainage class.
2. Within each drought year with at least 15 D2+ counties with a penalty, regress the drought penalty on surface organic matter, **per +1 percentage point** of organic matter:
   - **A.** organic matter only
   - **B.** + longitude + latitude
   - **C.** + longitude + latitude + slope
3. Year-bootstrap 95% ranges; the same three specifications on the placebo (non-D2+ county-years, years with at least 15 such rows).
4. **Decision rule (fixed now), per crop:** call it **"a clear association"** only if specification **C**'s drought-year range **excludes zero AND** its placebo range **includes zero**. Any other pattern is **"no clear effect"**. Specifications A and B are reported but do not decide.

## Part 2: EXPLORATORY: other soil properties
- Specification **B** (longitude + latitude) for **clay** (per +1 pp), **sand** (per +1 pp), **slope** (per +1 pp) and **dominant drainage class** (dummies for each class present, Well drained as reference), drought versus placebo, in one table labelled **EXPLORATORY**.
- **Test count.** Tests = drought-year estimates in Parts 1 and 2 (placebo estimates are checks, not tests, and are not counted): Part 1 has 3 specifications x 2 crops = 6; Part 2 has 3 continuous variables plus one dummy per non-reference drainage class present, x 2 crops. The notebook reports the total N and the number expected to exclude zero by chance alone at a 95% range, 0.05 x N. No single estimate from Part 2 is treated as a finding.

## Part 3: EXPLORATORY: conservation practices (2017 Census)
- Exposure: **2017** shares only (`no_till_share`, `cover_crop_share`), measured before the outcomes. Outcome: drought penalties in **drought years 2018–2025** that meet the 15-county rule (at least 15 D2+ counties with a penalty in that year). **If fewer than 3 such years qualify for a crop, report that and stop Part 3 for that crop.**
- Within each qualifying year, regress the penalty on the 2017 **no-till share (per +10 percentage points)** and, separately, the 2017 **cover-crop share (per +1 percentage point)**, each with longitude + latitude + slope. Same bootstrap; placebo = non-D2+ county-years in 2018–2025, years with at least 15 such rows.
- **Guardrail (fixed):** control only for variables practices cannot change (location, slope). **Never** control for organic matter, `aws_100cm_mm` or any other soil property that practices may affect; those lie on the pathway from practice to drought loss (Bueno de Mesquita & Fowler, ch. 10; Gelman, Hill & Vehtari, §19.6).
- Interpretation written in advance: counties may have adopted practices **because** of earlier droughts (for example 2012), so even the direction of any association is ambiguous (Bueno de Mesquita & Fowler, ch. 9); farmers choose these practices, so no association here shows a practice effect.

## Summary rules (written after the results, judged by the rules above)
- Part 1: apply the decision rule and state which outcome it gives for each crop.
- If any range excludes zero, say that with this few years the size is likely overstated and should not be quoted as the effect (Gelman, Hill & Vehtari, §4.4, §16.1).
- Parts 2 and 3: patterns only; nothing is claimed from a single estimate.
- Compare with the storage result from notebook 05 (`penalty_storage_results` on `aws_100cm_mm`, recomputed here with the unchanged module settings as the reference row).

## Guardrails
- No specification, unit, threshold, year rule or control set is changed after seeing results; everything above is reported in full.
- `soil_planner/soil_tool.py`, the app, the priority ranking and the evidence tables are not changed. The extra soil and practice data remain display-only everywhere else.
- Nothing under `data/` or `src/` is modified. The notebook runs from the repo root with `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 notebooks/07_soil_and_practices.ipynb`.
